# Dual-Stream Deepfake Forensics and Human Activity Recognition — Reproducible Experiments

This notebook rebuilds the lightweight six-block CNN used by both streams of the dual-stream framework and evaluates it on public data. Every result reported in the accompanying article is produced here.

| Part | What it does | Main outputs |
|---|---|---|
| **A** | Deepfake stream: two training recipes on *140k Real and Fake Faces* — the **standard** recipe (flip, rotation, zoom) and a **robust** recipe (adds JPEG compression, blur, rescaling, colour jitter). Held-out test evaluation. | accuracy, precision, sensitivity, specificity, F1, AUC with bootstrap 95% CIs; confusion matrices |
| **B** | Cross-dataset generalisation on *Hard Fake vs Real Faces* (no retraining), with raw and face-aligned preprocessing, AUC, and two decision thresholds fixed on in-domain validation data | cross-dataset table, ROC curves |
| **C** | Single vs. fused deep features (AlexNet, VGG16, ResNet50 + linear SVM), repeated runs with a sequential stopping rule, Wilcoxon and McNemar tests | fusion table |
| **D** | HAR stream on the *Human Activity Recognition – Video Dataset*, split **by video** | frame- and video-level accuracy, confusion matrix |
| **E** | Full dual-stream pipeline on real video: per-stage timing for a naïve and an optimised pipeline (threaded decoding, YuNet face detection on a reduced frame, detect-and-hold, HAR on every 2nd frame) | stage breakdown, latency, throughput, real-time verdict |
| **F** | Export of every table as LaTeX and JSON, figures, and a backup of the (small) results folder to Drive | `results_tables.tex`, `results.json`, `figures/` |

### Design rules applied throughout
1. **No silent cells.** Every long step prints its progress and an estimate of the remaining time.
2. **Every iterative loop stops by an explicit rule.** Training stops when validation accuracy has not improved by at least `min_delta` over `patience` epochs; repeated runs, bootstrap resampling and timing loops stop as soon as their estimates have converged (confidence-interval width below a stated tolerance), with a minimum and a maximum number of iterations. Each loop prints its stopping status.
3. **No tuning on test data.** All choices (augmentations, thresholds, face-alignment geometry) are fixed in advance or estimated on in-domain training/validation data only.
4. **Resumable.** Trained models and per-epoch checkpoints are stored on Drive (`MyDrive/Outputs/DualStream/checkpoints`, a few MB). A disconnected session resumes where it stopped; a finished model is reloaded, not retrained.
5. **Storage.** Datasets are read from `MyDrive/Datasets/`. Every result — `results.json` (updated after each part), LaTeX tables, figures, model checkpoints — is written **directly to Drive** (`MyDrive/Outputs/DualStream/`, a few MB in total), so nothing is lost if the session ends. Only large temporary data (extracted archives, image caches, extracted features, video frames; tens of GB) stays on the Colab disk `/content/work`, which is erased when the session ends.

**Runtime:** GPU (A100 recommended). `TIER = "full"` produces the reported results; `TIER = "quick"` runs every part on subsets for an end-to-end check.

## 0. Configuration

In [ ]:
# =============================== USER SETTINGS ===============================
TIER = "full"            # "full" = reported results | "quick" = subsets, end-to-end check | "smoke" = tiny test

DRIVE_ROOT   = "/content/drive/MyDrive"
DATASETS_DIR = f"{DRIVE_ROOT}/Datasets"
HAR_SRC      = f"{DATASETS_DIR}/HAR/HAR_datasets.zip"          # Human Activity Recognition - Video Dataset
FACES_SRC    = f"{DATASETS_DIR}/HAR/Real_Fake_Faces.zip"       # 140k Real and Fake Faces
EXTERNAL_SRC = {"Hard Fake vs Real Faces": f"{DATASETS_DIR}/HAR/Hard_Fake_Real_Faces.zip"}

WORK = "/content/work"                                    # LARGE temporary data only: extracted archives, image caches, features
OUT  = f"{DRIVE_ROOT}/Outputs/DualStream/results"      # results.json, LaTeX tables, figures -> written directly to Drive
CKPT = f"{DRIVE_ROOT}/Outputs/DualStream/checkpoints"     # model checkpoints (a few MB) -> Drive
SEED = 42
# =============================================================================

import os, sys, json, time, random, shutil, zipfile, platform, subprocess, math
from pathlib import Path
import numpy as np
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

# ---- Mount Google Drive FIRST, before any folder under /content/drive is touched ----
if "google.colab" in sys.modules or os.path.exists("/content"):
    try:
        from google.colab import drive
        if not os.path.ismount("/content/drive"):
            if os.path.exists("/content/drive"):           # leftover local folder would block the mount
                shutil.rmtree("/content/drive")
            drive.mount("/content/drive")
    except ImportError:
        pass                                              # not running in Colab
assert os.path.isdir(DATASETS_DIR), f"Drive not mounted or datasets folder missing: {DATASETS_DIR}"
for zp in [HAR_SRC, FACES_SRC] + list(EXTERNAL_SRC.values()):
    print(("  found   " if os.path.exists(zp) else "  MISSING ") + zp)

for d in (WORK, OUT, f"{OUT}/figures"):
    Path(d).mkdir(parents=True, exist_ok=True)

# Every iterative procedure has a minimum, a maximum and a convergence tolerance.
T = dict(
    full=dict(
        faces_cap=None,                                   # use the whole 140k dataset
        train=dict(max_epochs=275, patience=8, min_delta=0.005),       # stop: <0.5 pt gain over 8 epochs
        boot=dict(min_n=1000, max_n=10000, step=500, tol=0.0005),      # stop: CI bounds move <0.05 pt
        fusion=dict(n_train=20000, n_test=5000, min_runs=6, max_runs=15, ci_tol=0.003),  # CI half-width <0.3 pt
        har_frames=16,
        timing=dict(warm=50, min_frames=200, max_frames=2000, rel_ci=0.02),  # stop: mean known to ±2%
        align_ref=500,                                    # in-domain images used to estimate face geometry
    ),
    quick=dict(
        faces_cap=(20000, 4000, 4000),
        train=dict(max_epochs=30, patience=5, min_delta=0.005),
        boot=dict(min_n=500, max_n=3000, step=250, tol=0.001),
        fusion=dict(n_train=4000, n_test=2000, min_runs=6, max_runs=10, ci_tol=0.005),
        har_frames=8,
        timing=dict(warm=20, min_frames=100, max_frames=500, rel_ci=0.03),
        align_ref=200,
    ),
    smoke=dict(
        faces_cap=(64, 32, 32),
        train=dict(max_epochs=4, patience=2, min_delta=0.005),
        boot=dict(min_n=100, max_n=300, step=100, tol=0.01),
        fusion=dict(n_train=48, n_test=32, min_runs=3, max_runs=4, ci_tol=0.05),
        har_frames=4,
        timing=dict(warm=3, min_frames=10, max_frames=30, rel_ci=0.2),
        align_ref=20,
    ),
)[TIER]
print("TIER:", TIER)
print(json.dumps(T, indent=1))

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s)
    import tensorflow as tf; tf.random.set_seed(s)
set_seed()

RESULTS = {"tier": TIER, "seed": SEED, "settings": T}
def save_results():
    """Write results.json (called after every part, so partial results survive a disconnect)."""
    json.dump(RESULTS, open(f"{OUT}/results.json", "w"), indent=2, default=float)

class Timer:
    """Context manager that prints how long a step took."""
    def __init__(self, label): self.label = label
    def __enter__(self): self.t0 = time.time(); print(f"▶ {self.label} ...", flush=True); return self
    def __exit__(self, *a): print(f"✔ {self.label}: {(time.time() - self.t0) / 60:.1f} min", flush=True)

In [ ]:
# ---- Environment: install what is missing, record hardware ----
try:
    import cv2
    assert hasattr(cv2, "FaceDetectorYN") and hasattr(cv2, "CascadeClassifier")
except Exception:                                  # OpenCV 4.8+ provides YuNet; 4.x provides Haar cascades
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "opencv-python-headless>=4.8,<5"], check=False)
    import cv2
import tensorflow as tf
gpus = tf.config.list_physical_devices("GPU")
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)   # leave GPU memory for PyTorch and cuML
gpu_name = "none"
try:
    gpu_name = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                              capture_output=True, text=True).stdout.strip() or "none"
except Exception:
    pass
cpu_name = platform.processor()
try:
    cpu_name = [l.split(":")[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name")][0]
except Exception:
    pass
RESULTS["env"] = dict(python=platform.python_version(), tensorflow=tf.__version__, opencv=cv2.__version__,
                      gpu=gpu_name, cpu=cpu_name, cpu_count=os.cpu_count())
print(json.dumps(RESULTS["env"], indent=1))

## 1. Data
The three archives are read from `MyDrive/Datasets/HAR/` and extracted once to `/content/work`. Labels are taken from folder names (`real` / `fake`); the 140k dataset's own `train` / `valid` / `test` folders define the split.

In [ ]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def extract_zip(zip_path, name):
    """Extract a zip from Drive to /content/work/<name> once; later calls reuse the extracted copy."""
    dst = Path(WORK) / name
    done = dst / ".extracted"
    if done.exists():
        print(f"  {name}: already extracted"); return dst
    zip_path = Path(zip_path)
    assert zip_path.exists(), f"missing dataset archive: {zip_path}"
    dst.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        members = z.infolist(); n = len(members); t0 = time.time()
        for i, m in enumerate(members, 1):
            z.extract(m, dst)
            if i % max(1, n // 10) == 0 or i == n:
                el = time.time() - t0
                print(f"  {name}: {i}/{n} files, {el / 60:.1f} min, ~{el / i * (n - i) / 60:.1f} min left", flush=True)
    done.touch()
    return dst

def list_images(root):
    return sorted(p for p in Path(root).rglob("*") if p.suffix.lower() in IMG_EXT)

def label_from_path(p):
    """0 = real, 1 = fake, None = unknown; taken from the nearest folder whose name says real or fake."""
    for s in reversed([s.lower() for s in Path(p).parts[:-1]]):
        if "real" in s and "fake" not in s: return 0
        if "fake" in s and "real" not in s: return 1
    return None

def split_of(p):
    parts = [s.lower() for s in Path(p).parts]
    for s in ("train", "valid", "val", "test"):
        if s in parts: return {"val": "valid"}.get(s, s)
    return None

with Timer("140k Real and Fake Faces"):
    faces_root = extract_zip(FACES_SRC, "faces140k")
    FACES = {"train": [], "valid": [], "test": []}
    for p in list_images(faces_root):
        s, y = split_of(p), label_from_path(p)
        if s in FACES and y is not None:
            FACES[s].append((str(p), y))
    rng = np.random.default_rng(SEED)
    for i, s in enumerate(("train", "valid", "test")):
        rng.shuffle(FACES[s])
        if T["faces_cap"]: FACES[s] = FACES[s][:T["faces_cap"][i]]
RESULTS["faces_split"] = {s: dict(n=len(v), fake=int(sum(y for _, y in v))) for s, v in FACES.items()}
print(RESULTS["faces_split"])
assert all(len(v) for v in FACES.values()), "train/valid/test folders with real/fake labels not found"
save_results()

## 2. Model and statistical utilities
**Network.** Six convolutional blocks (32 filters, then five × 64; 3×3 kernels; ReLU; 2×2 max pooling), flatten, dropout 0.25, dense 64, softmax. The parameter and FLOP counts are checked against the values reported in the article (183,682 parameters with a two-class head; ≈1.03 GFLOPs per 256×256×3 frame).

**Convergence utilities.** `bootstrap_ci` adds resamples in blocks until the 95% interval bounds move by less than `tol` between blocks. `PlateauStop` is the training stopping rule (it reads the full epoch history, so a resumed session continues the same count).

In [ ]:
from tensorflow.keras import layers, models
IMG = 256

def build_cnn(n_out):
    """Six-block CNN; the output layer stays float32 under mixed precision."""
    m = models.Sequential([layers.Input((IMG, IMG, 3))])
    for f in (32, 64, 64, 64, 64, 64):
        m.add(layers.Conv2D(f, 3, activation="relu", kernel_initializer="he_normal"))
        m.add(layers.MaxPooling2D(2))
    m.add(layers.Flatten()); m.add(layers.Dropout(0.25))
    m.add(layers.Dense(64, activation="relu"))
    m.add(layers.Dense(n_out, activation="softmax", dtype="float32"))
    return m

def count_flops(model):
    """FLOPs per frame from the TensorFlow profiler on the frozen inference graph."""
    from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2
    f = tf.function(lambda x: model(x)).get_concrete_function(tf.TensorSpec([1, IMG, IMG, 3], tf.float32))
    g = convert_variables_to_constants_v2(f).graph
    opts = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation(); opts["output"] = "none"
    return tf.compat.v1.profiler.profile(graph=g, run_meta=tf.compat.v1.RunMetadata(), cmd="op",
                                         options=opts).total_float_ops

tf.get_logger().setLevel("ERROR")
_m = build_cnn(2)
RESULTS["model"] = dict(params=int(_m.count_params()), flops=int(count_flops(_m)))
print(RESULTS["model"]); assert RESULTS["model"]["params"] == 183682; del _m

# ---------------------------- metrics ----------------------------
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve

def binary_metrics(y, yhat, score=None):
    """Fake = positive class. Returns counts and rates (and AUC when scores are given)."""
    y, yhat = np.asarray(y), np.asarray(yhat)
    tp = int(((y == 1) & (yhat == 1)).sum()); tn = int(((y == 0) & (yhat == 0)).sum())
    fp = int(((y == 0) & (yhat == 1)).sum()); fn = int(((y == 1) & (yhat == 0)).sum())
    prec = tp / max(tp + fp, 1); rec = tp / max(tp + fn, 1)
    out = dict(TP=tp, TN=tn, FP=fp, FN=fn, accuracy=(tp + tn) / len(y), precision=prec,
               recall_sensitivity=rec, specificity=tn / max(tn + fp, 1),
               f1=2 * prec * rec / max(prec + rec, 1e-12))
    if score is not None and len(set(y.tolist())) == 2:
        out["auc"] = float(roc_auc_score(y, score))
    return out

def bootstrap_ci(y, yhat, score=None, cfg=None, seed=SEED, label=""):
    """95% percentile bootstrap CIs. Resamples are added in blocks of cfg['step'] until every bound moves by
    less than cfg['tol'] between consecutive blocks (after cfg['min_n']), or cfg['max_n'] is reached."""
    cfg = cfg or T["boot"]; rng = np.random.default_rng(seed)
    y, yhat = np.asarray(y), np.asarray(yhat); s = None if score is None else np.asarray(score)
    keys = ["accuracy", "precision", "recall_sensitivity", "specificity", "f1"] + (["auc"] if s is not None else [])
    draws, prev = [], None
    while True:
        for _ in range(cfg["step"]):
            i = rng.integers(0, len(y), len(y))
            m = binary_metrics(y[i], yhat[i], None if s is None else s[i])
            draws.append([m.get(k, np.nan) for k in keys])
        lo, hi = np.nanpercentile(draws, [2.5, 97.5], axis=0); cur = np.r_[lo, hi]
        n = len(draws)
        if prev is not None and n >= cfg["min_n"]:
            shift = float(np.max(np.abs(cur - prev)))
            if shift < cfg["tol"] or n >= cfg["max_n"]:
                why = "converged" if shift < cfg["tol"] else "maximum reached"
                print(f"  bootstrap {label}: {n} resamples, largest bound shift {100*shift:.3f} pts -> {why}")
                break
        prev = cur
    return {k: (float(a), float(b)) for k, a, b in zip(keys, lo, hi)}, n

## 3. Input pipelines and training
**Standard recipe** (as in the article): random horizontal flip, ±10° rotation, ±10% zoom.

**Robust recipe** (fixed in advance, following common practice for detectors meant to generalise across generators): the standard recipe plus, each applied independently,
Gaussian blur (σ ∈ [0.1, 3], p = 0.5), JPEG re-compression (quality ∈ [30, 100], p = 0.5), down-and-up rescaling (factor ∈ [0.5, 1], p = 0.3), and colour jitter (brightness ±0.1, contrast and saturation ×[0.8, 1.2], p = 0.5).

**Training.** Adam (lr 1e-4), batch 32, categorical cross-entropy, mixed precision on GPU. Decoded images are cached on local disk after the first epoch. The checkpoint with the best validation accuracy is kept; training stops by the plateau rule printed after every epoch.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
geom_aug = tf.keras.Sequential([layers.RandomFlip("horizontal"),
                                layers.RandomRotation(10 / 360), layers.RandomZoom(0.1)])

def _decode(path, y, n_cls):
    x = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    x = tf.cast(tf.image.resize(x, (IMG, IMG)), tf.uint8)      # cached as uint8 (4x smaller than float)
    return x, tf.one_hot(y, n_cls)

def _gaussian_blur(x, sigma):
    r = tf.range(-4.0, 5.0)
    g = tf.exp(-(r ** 2) / (2.0 * sigma ** 2)); g = g / tf.reduce_sum(g)
    k = tf.tile((g[:, None] * g[None, :])[:, :, None, None], [1, 1, 3, 1])
    return tf.nn.depthwise_conv2d(x[None], k, [1, 1, 1, 1], "SAME")[0]

def robust_aug(x):
    """Photometric/compression augmentations of the robust recipe; x is one float image in [0, 1]."""
    u = lambda: tf.random.uniform([])
    x = tf.cond(u() < 0.5, lambda: _gaussian_blur(x, tf.random.uniform([], 0.1, 3.0)), lambda: x)
    x = tf.cond(u() < 0.5, lambda: tf.image.adjust_jpeg_quality(x, tf.random.uniform([], 30, 101, tf.int32)), lambda: x)
    def rescale(x):
        s = tf.random.uniform([], 0.5, 1.0); n = tf.cast(s * IMG, tf.int32)
        return tf.image.resize(tf.image.resize(x, (n, n)), (IMG, IMG))
    x = tf.cond(u() < 0.3, lambda: rescale(x), lambda: x)
    def jitter(x):
        x = tf.image.random_brightness(x, 0.1); x = tf.image.random_contrast(x, 0.8, 1.2)
        return tf.image.random_saturation(x, 0.8, 1.2)
    x = tf.cond(u() < 0.5, lambda: jitter(x), lambda: x)
    return tf.clip_by_value(tf.reshape(x, (IMG, IMG, 3)), 0.0, 1.0)

def make_ds(pairs, n_cls, train=False, recipe="standard", batch=32, cache=None):
    paths = [p for p, _ in pairs]; ys = [int(y) for _, y in pairs]
    ds = tf.data.Dataset.from_tensor_slices((paths, ys)).map(lambda p, y: _decode(p, y, n_cls), AUTOTUNE)
    if cache:
        Path(f"{WORK}/cache").mkdir(exist_ok=True); ds = ds.cache(f"{WORK}/cache/{cache}")
    if train:
        ds = ds.shuffle(min(len(paths), 5000), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(lambda x, y: (tf.cast(x, tf.float32) / 255.0, y), AUTOTUNE)
    if train and recipe == "robust":
        ds = ds.map(lambda x, y: (robust_aug(x), y), AUTOTUNE)
    ds = ds.batch(batch)
    if train:
        ds = ds.map(lambda x, y: (geom_aug(x, training=True), y), AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

class PlateauStop(tf.keras.callbacks.Callback):
    """Stop when the best validation accuracy of the last `patience` epochs exceeds the best value reached
    before them by less than `min_delta`. Reads the whole history file, so resumed sessions keep the count."""
    def __init__(self, hist_file, patience, min_delta):
        super().__init__(); self.f, self.patience, self.min_delta = Path(hist_file), patience, min_delta
    def on_epoch_end(self, epoch, logs=None):
        import pandas as pd
        acc = []
        if self.f.exists():
            h = pd.read_csv(self.f).drop_duplicates("epoch", keep="last").sort_values("epoch")
            acc = [a for e, a in zip(h["epoch"], h["val_accuracy"]) if e < epoch]
        acc.append(float(logs["val_accuracy"]))
        msg = f"  [stop rule] epoch {epoch + 1}: val_acc {acc[-1]:.4f}, best {max(acc):.4f}"
        if len(acc) > self.patience:
            gain = max(acc) - max(acc[:-self.patience])
            msg += f", gain over last {self.patience} epochs {100 * gain:.2f} pts (needs {100 * self.min_delta:.2f})"
            if gain < self.min_delta:
                print(msg + " -> STOP", flush=True); self.model.stop_training = True; return
        print(msg, flush=True)

def train_cnn(train_pairs, val_pairs, n_cls, tag, recipe="standard"):
    """Train (or reload) a model. Per-epoch checkpoints live on Drive; a finished model is reloaded."""
    import pandas as pd
    ck = Path(CKPT); ck.mkdir(parents=True, exist_ok=True)
    final, best = ck / f"{tag}_final.weights.h5", ck / f"{tag}_best.weights.h5"
    info_f, hist_f = ck / f"{tag}_info.json", ck / f"{tag}_history.csv"
    if final.exists() and info_f.exists():
        print(f"[{tag}] finished model found on Drive: reloading, no retraining")
        tf.keras.mixed_precision.set_global_policy("float32")
        m = build_cnn(n_cls); m.load_weights(final)
        return m, pd.read_csv(hist_f).drop_duplicates("epoch", keep="last").to_dict("list"), json.load(open(info_f))
    set_seed()
    if gpus: tf.keras.mixed_precision.set_global_policy("mixed_float16")
    model = build_cnn(n_cls)
    model.compile(tf.keras.optimizers.Adam(1e-4, beta_1=0.9, beta_2=0.999),
                  loss="categorical_crossentropy", metrics=["accuracy"])
    cfg = T["train"]
    cbs = [tf.keras.callbacks.BackupAndRestore(str(ck / f"{tag}_backup"), save_freq="epoch", delete_checkpoint=False),
           tf.keras.callbacks.ModelCheckpoint(str(best), monitor="val_accuracy", save_best_only=True, save_weights_only=True),
           tf.keras.callbacks.CSVLogger(str(hist_f), append=True),
           PlateauStop(hist_f, cfg["patience"], cfg["min_delta"])]
    t0 = time.time()
    print(f"[{tag}] training, recipe = {recipe}, up to {cfg['max_epochs']} epochs (plateau rule above)", flush=True)
    model.fit(make_ds(train_pairs, n_cls, True, recipe, cache=f"{tag}_train"),
              validation_data=make_ds(val_pairs, n_cls, cache=f"{tag}_valid"),
              epochs=cfg["max_epochs"], callbacks=cbs, verbose=1)
    hist = pd.read_csv(hist_f).drop_duplicates("epoch", keep="last").to_dict("list")
    model.load_weights(best)
    tf.keras.mixed_precision.set_global_policy("float32")
    m32 = build_cnn(n_cls); m32.set_weights(model.get_weights()); m32.save_weights(final)
    info = dict(epochs_run=len(hist["loss"]), best_epoch=int(np.argmax(hist["val_accuracy"]) + 1),
                best_val_acc=float(np.max(hist["val_accuracy"])), recipe=recipe,
                train_minutes_this_session=(time.time() - t0) / 60, mixed_precision=bool(gpus))
    json.dump(info, open(info_f, "w"))
    return m32, hist, info

def predict_probs(model, pairs, n_cls, label=""):
    """Class probabilities for a list of (path, label) pairs, with progress."""
    ds = make_ds(pairs, n_cls, batch=128); n = math.ceil(len(pairs) / 128); out = []
    for i, (x, _) in enumerate(ds, 1):
        out.append(model(x, training=False).numpy())
        if i % 50 == 0 or i == n: print(f"  predicting {label}: batch {i}/{n}", flush=True)
    return np.concatenate(out)

## 4. Part A — deepfake stream: standard and robust recipes
The standard model (`deepfake`) is reloaded from its checkpoint if one exists. The robust model (`deepfake_robust`) is trained with the robust recipe under the same stopping rule. Both are evaluated once on the 20,000-image test split. A decision threshold is also fixed on the **validation** split (the value maximising balanced accuracy); it is used later for cross-dataset testing.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import balanced_accuracy_score

MODELS, HIST = {}, {}
RESULTS["A_deepfake"] = {}
for tag, recipe in (("deepfake", "standard"), ("deepfake_robust", "robust")):
    with Timer(f"Part A — {tag}"):
        model, hist, info = train_cnn(FACES["train"], FACES["valid"], 2, tag, recipe)
        MODELS[tag], HIST[tag] = model, hist
        # threshold fixed on validation data only
        pv = predict_probs(model, FACES["valid"], 2, f"{tag} / valid")[:, 1]
        yv = np.array([y for _, y in FACES["valid"]])
        grid = np.linspace(0.01, 0.99, 99)
        thr_val = float(grid[np.argmax([balanced_accuracy_score(yv, pv >= t) for t in grid])])
        # held-out test evaluation
        pt = predict_probs(model, FACES["test"], 2, f"{tag} / test")[:, 1]
        yt = np.array([y for _, y in FACES["test"]]); yh = (pt >= 0.5).astype(int)
        m = binary_metrics(yt, yh, pt)
        ci, nboot = bootstrap_ci(yt, yh, pt, label=tag)
        RESULTS["A_deepfake"][tag] = dict(train=info, recipe=recipe, threshold_val=thr_val, test=m,
                                          ci95=ci, bootstrap_resamples=nboot)
        np.save(f"{OUT}/A_{tag}_test_scores.npy", np.c_[yt, pt])
        print(f"  {tag}: acc {100*m['accuracy']:.2f}%, AUC {m['auc']:.4f}, sens {100*m['recall_sensitivity']:.2f}%, "
              f"spec {100*m['specificity']:.2f}%, validation threshold {thr_val:.2f}")
    save_results()

fig, ax = plt.subplots(2, 3, figsize=(15, 8))
for r, tag in enumerate(MODELS):
    h, m = HIST[tag], RESULTS["A_deepfake"][tag]["test"]
    ax[r, 0].plot(h["accuracy"], label="train"); ax[r, 0].plot(h["val_accuracy"], label="validation")
    ax[r, 0].set_title(f"{tag}: accuracy"); ax[r, 0].set_xlabel("epoch"); ax[r, 0].legend()
    ax[r, 1].plot(h["loss"], label="train"); ax[r, 1].plot(h["val_loss"], label="validation")
    ax[r, 1].set_title(f"{tag}: loss"); ax[r, 1].set_xlabel("epoch"); ax[r, 1].legend()
    cm = np.array([[m["TP"], m["FN"]], [m["FP"], m["TN"]]])
    ax[r, 2].imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2): ax[r, 2].text(j, i, cm[i, j], ha="center", va="center")
    ax[r, 2].set_xticks([0, 1], ["fake", "real"]); ax[r, 2].set_yticks([0, 1], ["fake", "real"])
    ax[r, 2].set_xlabel("Predicted"); ax[r, 2].set_ylabel("True"); ax[r, 2].set_title(f"{tag}: test confusion matrix")
plt.tight_layout(); plt.savefig(f"{OUT}/figures/A_curves_confusion.png", dpi=200); plt.show()

## 5. Part B — cross-dataset generalisation
Both models are applied **without retraining** to *Hard Fake vs Real Faces*. Two preprocessing variants are evaluated:

* **raw** — the image is resized to 256×256, exactly as in training;
* **face-aligned** — a YuNet face detector locates the face and the image is cropped so that the face occupies the same fraction and position of the frame as in the **training-domain** images (geometry estimated on 140k validation images only).

Results are reported with AUC (threshold-free) and at two thresholds fixed in advance: 0.5 and the validation threshold from Part A. Hard Fake data are never used to choose anything.

In [ ]:
import urllib.request

# ---------------- YuNet face detector (OpenCV model zoo) ----------------
YUNET = Path(WORK) / "face_detection_yunet_2023mar.onnx"
YUNET_URLS = [
    "https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx",
    "https://media.githubusercontent.com/media/opencv/opencv_zoo/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx",
]
def _valid_onnx(p):
    try:
        cv2.FaceDetectorYN.create(str(p), "", (320, 320)); return True
    except Exception:
        return False
if not (YUNET.exists() and _valid_onnx(YUNET)):
    for u in YUNET_URLS:
        try:
            urllib.request.urlretrieve(u, YUNET)
            if _valid_onnx(YUNET): print("YuNet downloaded from", u); break
        except Exception as e:
            print("  download failed:", u, e)
FACE_DETECTOR = "yunet" if YUNET.exists() and _valid_onnx(YUNET) else "haar"
print("face detector:", FACE_DETECTOR)
RESULTS["face_detector"] = FACE_DETECTOR

def detect_largest_face(img_bgr, width=None):
    """Return (x, y, w, h) of the largest face in original-image coordinates, or None."""
    h0, w0 = img_bgr.shape[:2]
    s = 1.0 if (width is None or w0 <= width) else width / w0
    small = img_bgr if s == 1.0 else cv2.resize(img_bgr, (int(w0 * s), int(h0 * s)))
    if FACE_DETECTOR == "yunet":
        det = cv2.FaceDetectorYN.create(str(YUNET), "", (small.shape[1], small.shape[0]), 0.6, 0.3, 50)
        _, faces = det.detect(small)
        if faces is None or len(faces) == 0: return None
        b = max(faces, key=lambda f: f[2] * f[3])[:4]
    else:
        casc = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
        f = casc.detectMultiScale(cv2.cvtColor(small, cv2.COLOR_BGR2GRAY), 1.2, 5)
        if len(f) == 0: return None
        b = max(f, key=lambda r: r[2] * r[3])
    return tuple(float(v) / s for v in b)

# ---------------- face geometry of the training domain (validation images only) ----------------
with Timer("face geometry of 140k validation images"):
    ratios, cx, cy = [], [], []
    for i, (p, _) in enumerate(FACES["valid"][:T["align_ref"]], 1):
        im = cv2.imread(p); b = detect_largest_face(im)
        if b:
            ratios.append(b[2] / im.shape[1]); cx.append((b[0] + b[2] / 2) / im.shape[1]); cy.append((b[1] + b[3] / 2) / im.shape[0])
        if i % 100 == 0: print(f"  {i}/{T['align_ref']} images, faces found in {len(ratios)}", flush=True)
GEOM = dict(face_width_ratio=float(np.median(ratios)), centre_x=float(np.median(cx)), centre_y=float(np.median(cy)),
            n=len(ratios))
print("training-domain face geometry:", GEOM)

def align_like_training(img_bgr):
    """Crop so that the detected face has the training-domain size and position; None if no face."""
    b = detect_largest_face(img_bgr)
    if b is None: return None
    side = b[2] / GEOM["face_width_ratio"]
    x0 = b[0] + b[2] / 2 - GEOM["centre_x"] * side; y0 = b[1] + b[3] / 2 - GEOM["centre_y"] * side
    pad = int(side)
    big = cv2.copyMakeBorder(img_bgr, pad, pad, pad, pad, cv2.BORDER_REFLECT)
    x0, y0 = int(round(x0 + pad)), int(round(y0 + pad)); s = int(round(side))
    return cv2.resize(big[y0:y0 + s, x0:x0 + s], (IMG, IMG), interpolation=cv2.INTER_AREA)

In [ ]:
RESULTS["B_cross"] = {}
ROC = {}
for ds_name, zp in EXTERNAL_SRC.items():
    if not Path(zp).exists():
        print(f"[skip] {ds_name}: {zp} not found"); continue
    root = extract_zip(zp, "ext_" + "".join(c for c in ds_name if c.isalnum()))
    pairs = [(str(p), label_from_path(p)) for p in list_images(root)]
    pairs = [(p, y) for p, y in pairs if y is not None]
    if T["faces_cap"]: pairs = pairs[:T["faces_cap"][2]]
    print(f"{ds_name}: {len(pairs)} images ({sum(y for _, y in pairs)} fake)")

    # face-aligned copies written to /content/work (images without a detected face keep the raw resize)
    with Timer(f"{ds_name}: face alignment"):
        adir = Path(WORK) / ("aligned_" + "".join(c for c in ds_name if c.isalnum())); adir.mkdir(exist_ok=True)
        aligned, n_face = [], 0
        for i, (p, y) in enumerate(pairs, 1):
            q = adir / f"{i:06d}.png"
            if not q.exists():
                a = align_like_training(cv2.imread(p))
                if a is None: a = cv2.resize(cv2.imread(p), (IMG, IMG))
                else: n_face += 1
                cv2.imwrite(str(q), a)
            else:
                n_face = -1                                   # reused from an earlier session
            aligned.append((str(q), y))
            if i % 200 == 0 or i == len(pairs): print(f"  {i}/{len(pairs)} images", flush=True)
    y = np.array([y for _, y in pairs])
    RESULTS["B_cross"][ds_name] = dict(n=len(pairs), n_fake=int(y.sum()), faces_detected=n_face, models={})
    for tag, model in MODELS.items():
        thr_val = RESULTS["A_deepfake"][tag]["threshold_val"]
        RESULTS["B_cross"][ds_name]["models"][tag] = {}
        for variant, plist in (("raw", pairs), ("aligned", aligned)):
            s = predict_probs(model, plist, 2, f"{tag} / {variant}")[:, 1]
            row = {"threshold_0.5": binary_metrics(y, (s >= 0.5).astype(int), s),
                   "threshold_val": binary_metrics(y, (s >= thr_val).astype(int), s), "thr_val": thr_val}
            RESULTS["B_cross"][ds_name]["models"][tag][variant] = row
            ROC[(ds_name, tag, variant)] = roc_curve(y, s)[:2]
            print(f"  {tag:16s} {variant:8s} AUC {row['threshold_0.5']['auc']:.4f} | "
                  f"acc@0.5 {100*row['threshold_0.5']['accuracy']:.2f}% (sens {100*row['threshold_0.5']['recall_sensitivity']:.1f}%) | "
                  f"acc@val-thr {100*row['threshold_val']['accuracy']:.2f}%")
    save_results()

if ROC:
    plt.figure(figsize=(6, 6))
    for (dn, tag, var), (fpr, tpr) in ROC.items():
        plt.plot(fpr, tpr, label=f"{tag}, {var}")
    plt.plot([0, 1], [0, 1], "k--", lw=0.8); plt.xlabel("False-positive rate"); plt.ylabel("True-positive rate")
    plt.title("Cross-dataset ROC (Hard Fake vs Real Faces)"); plt.legend(); plt.tight_layout()
    plt.savefig(f"{OUT}/figures/B_cross_roc.png", dpi=200); plt.show()

## 6. Part C — single vs. fused deep features
Features: AlexNet fc6/fc7, VGG16 fc6/fc7 (ImageNet weights, after ReLU) and ResNet50 fc1000, extracted once in a separate PyTorch process. Each configuration is classified by a linear SVM (squared hinge loss, L2 penalty, C = 1) on standardised features; on GPU the RAPIDS cuML implementation is used, otherwise scikit-learn.

**Sequential stopping.** Runs are added one at a time (each run draws a new class-balanced training subsample; the test subsample is fixed). After `min_runs`, the loop stops as soon as the 95% confidence half-width of the mean accuracy is below `ci_tol` for every configuration (or at `max_runs`). We report mean ± s.d., the paired Wilcoxon signed-rank test of each fused configuration against the best single feature set, and McNemar's test on the first run.

In [ ]:
def balanced_idx(pairs, n, seed):
    rng = np.random.default_rng(seed); out = []
    for c in (0, 1):
        idx = [i for i, (_, y) in enumerate(pairs) if y == c]; rng.shuffle(idx); out += idx[: n // 2]
    return sorted(out)

FC = T["fusion"]
pool = [FACES["train"][i] for i in balanced_idx(FACES["train"], min(len(FACES["train"]), 2 * FC["n_train"]), SEED)]
ftest = [FACES["test"][i] for i in balanced_idx(FACES["test"], min(len(FACES["test"]), FC["n_test"]), SEED)]
json.dump({"pool": pool, "test": ftest}, open(f"{WORK}/fusion_lists.json", "w"))

FEAT_SCRIPT = r"""
import sys, json, numpy as np, torch, torchvision
from torchvision import transforms as TV
from PIL import Image
W = sys.argv[1]; L = json.load(open(f"{W}/fusion_lists.json"))
dev = "cuda" if torch.cuda.is_available() else "cpu"
tfm = TV.Compose([TV.Resize((224, 224)), TV.ToTensor(), TV.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
M = torchvision.models
alex = M.alexnet(weights=M.AlexNet_Weights.IMAGENET1K_V1).eval().to(dev)
vgg = M.vgg16(weights=M.VGG16_Weights.IMAGENET1K_V1).eval().to(dev)
res = M.resnet50(weights=M.ResNet50_Weights.IMAGENET1K_V1).eval().to(dev)
class DS(torch.utils.data.Dataset):
    def __init__(s, p): s.p = p
    def __len__(s): return len(s.p)
    def __getitem__(s, i): return tfm(Image.open(s.p[i][0]).convert("RGB"))
KEYS = ["AlexNet fc6", "AlexNet fc7", "VGG16 fc6", "VGG16 fc7", "ResNet50 fc1000"]
for part in ("pool", "test"):
    F = {k: [] for k in KEYS}; nb = (len(L[part]) + 63) // 64
    with torch.no_grad():
        for bi, x in enumerate(torch.utils.data.DataLoader(DS(L[part]), batch_size=64, num_workers=4), 1):
            x = x.to(dev)
            a = torch.flatten(alex.avgpool(alex.features(x)), 1)
            a6 = alex.classifier[2](alex.classifier[1](a)); a7 = alex.classifier[5](alex.classifier[4](a6))
            v = torch.flatten(vgg.avgpool(vgg.features(x)), 1)
            v6 = vgg.classifier[1](vgg.classifier[0](v)); v7 = vgg.classifier[4](vgg.classifier[3](v6))
            for k, t in zip(KEYS, (a6, a7, v6, v7, res(x))): F[k].append(t.cpu().numpy().astype(np.float32))
            if bi % 50 == 0 or bi == nb: print(f"  features [{part}] batch {bi}/{nb}", flush=True)
    np.savez(f"{W}/features_{part}.npz", **{k: np.concatenate(v) for k, v in F.items()})
"""
with Timer("Part C — feature extraction"):
    if Path(f"{WORK}/features_test.npz").exists():
        print("  features already extracted: reusing them")
    else:
        open(f"{WORK}/extract_features.py", "w").write(FEAT_SCRIPT)
        r = subprocess.run([sys.executable, f"{WORK}/extract_features.py", WORK], text=True)
        assert r.returncode == 0, "feature extraction failed"
F_pool = dict(np.load(f"{WORK}/features_pool.npz")); F_test = dict(np.load(f"{WORK}/features_test.npz"))
y_pool = np.array([y for _, y in pool]); y_ft = np.array([y for _, y in ftest])
print({k: v.shape for k, v in F_pool.items()})

In [ ]:
from scipy.stats import wilcoxon, binomtest
from sklearn.svm import LinearSVC
try:
    from cuml.svm import LinearSVC as GPU_SVC
    SVM_BACKEND = "cuML (GPU)"
except Exception:
    GPU_SVC, SVM_BACKEND = None, "scikit-learn (CPU)"
print("SVM backend:", SVM_BACKEND)

CONFIGS = {
    "AlexNet fc6": ["AlexNet fc6"], "VGG16 fc6": ["VGG16 fc6"], "ResNet50 fc1000": ["ResNet50 fc1000"],
    "AlexNet fc6 + fc7": ["AlexNet fc6", "AlexNet fc7"], "VGG16 fc6 + fc7": ["VGG16 fc6", "VGG16 fc7"],
    "AlexNet fc6 + VGG16 fc6": ["AlexNet fc6", "VGG16 fc6"],
    "AlexNet fc6 + ResNet50 fc1000": ["AlexNet fc6", "ResNet50 fc1000"],
    "VGG16 fc6 + ResNet50 fc1000": ["VGG16 fc6", "ResNet50 fc1000"],
    "AlexNet fc6 + VGG16 fc6 + ResNet50 fc1000": ["AlexNet fc6", "VGG16 fc6", "ResNet50 fc1000"],
}
SINGLE = ["AlexNet fc6", "VGG16 fc6", "ResNet50 fc1000"]

def svm_accuracy(Xtr, ytr, Xte, yte, seed):
    mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-8
    Xtr, Xte = ((Xtr - mu) / sd).astype(np.float32), ((Xte - mu) / sd).astype(np.float32)
    clf = GPU_SVC(C=1.0, max_iter=5000, output_type="numpy") if GPU_SVC else \
          LinearSVC(C=1.0, dual=False, max_iter=5000, random_state=seed)
    clf.fit(Xtr, ytr); yh = np.asarray(clf.predict(Xte)).astype(int)
    return float((yh == yte).mean()), yh

acc = {k: [] for k in CONFIGS}; pred0 = {}
n_run_train = min(FC["n_train"], len(pool)); t0 = time.time()
for r in range(FC["max_runs"]):
    idx = balanced_idx(pool, n_run_train, 1000 + r)
    for name, keys in CONFIGS.items():
        a, yh = svm_accuracy(np.hstack([F_pool[k][idx] for k in keys]), y_pool[idx],
                             np.hstack([F_test[k] for k in keys]), y_ft, r)
        acc[name].append(a)
        if r == 0: pred0[name] = yh
    n = r + 1
    half = {k: 1.96 * np.std(v, ddof=1) / np.sqrt(n) if n > 1 else np.inf for k, v in acc.items()}
    worst = max(half.values())
    print(f"run {n}: " + ", ".join(f"{k}={v[-1]:.4f}" for k, v in acc.items()), flush=True)
    print(f"  [stop rule] after {n} runs ({(time.time()-t0)/60:.1f} min): largest 95% CI half-width "
          f"{100*worst:.2f} pts (needs < {100*FC['ci_tol']:.2f}, min {FC['min_runs']} runs)", flush=True)
    if n >= FC["min_runs"] and worst < FC["ci_tol"]:
        print("  -> STOP: all means have converged"); break

best = max(SINGLE, key=lambda k: np.mean(acc[k]))
def mcnemar(a, b, y):
    ca, cb = a == y, b == y; n01 = int((ca & ~cb).sum()); n10 = int((~ca & cb).sum())
    return n01, n10, (binomtest(n01, n01 + n10, 0.5).pvalue if n01 + n10 else 1.0)
rows = {}
for name in CONFIGS:
    row = dict(mean=float(np.mean(acc[name])), sd=float(np.std(acc[name], ddof=1)), runs=acc[name])
    if name not in SINGLE:
        d = np.array(acc[name]) - np.array(acc[best])
        row["wilcoxon_p"] = float(wilcoxon(acc[name], acc[best]).pvalue) if np.any(d != 0) else 1.0
        row["mean_diff_vs_best"] = float(d.mean())
        row["mcnemar_n01_n10_p"] = mcnemar(pred0[name], pred0[best], y_ft)
    rows[name] = row
RESULTS["C_fusion"] = dict(best_single=best, n_train_per_run=n_run_train, n_test=len(ftest), runs=len(acc[best]),
                           svm_backend=SVM_BACKEND, configs=rows)
print(f"\nbest single feature set: {best}")
for k, v in rows.items():
    p = v.get("wilcoxon_p")
    print(f"{k:45s} {100*v['mean']:.2f} ± {100*v['sd']:.2f}" + ("" if p is None else f"   Wilcoxon p = {p:.4f}"))
save_results()

## 7. Part D — HAR stream
Videos are split **by video** (stratified 70/15/15, seed 42), so frames of one clip never appear in two subsets. `har_frames` frames per video are sampled uniformly in time (parallel extraction, progress printed). The same CNN with a seven-class head is trained under the same stopping rule; we report frame-level accuracy and video-level accuracy (mean of the frame probabilities).

In [ ]:
from sklearn.model_selection import train_test_split
from concurrent.futures import ThreadPoolExecutor, as_completed

with Timer("HAR video dataset"):
    har_root = extract_zip(HAR_SRC, "har")
videos = sorted(p for p in Path(har_root).rglob("*") if p.suffix.lower() in {".mp4", ".avi", ".mov", ".mkv"})
CLASSES = sorted({p.parent.name for p in videos}); cidx = {c: i for i, c in enumerate(CLASSES)}
print(len(videos), "videos;", {c: sum(p.parent.name == c for p in videos) for c in CLASSES})
vy = [cidx[p.parent.name] for p in videos]
v_tr, v_tmp, _, y_tmp = train_test_split(videos, vy, test_size=0.30, stratify=vy, random_state=SEED)
v_va, v_te = train_test_split(v_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)
cap = cv2.VideoCapture(str(v_te[0]))
VIDEO_INFO = dict(width=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), height=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
                  fps=float(cap.get(cv2.CAP_PROP_FPS))); cap.release()
print("example test video:", VIDEO_INFO)

FR = Path(WORK) / "har_frames"
def sample_frames(video, k):
    cap = cv2.VideoCapture(str(video)); n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); out = []
    for t in np.linspace(0, max(n - 1, 0), k).astype(int):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(t)); ok, fr = cap.read()
        if ok: out.append(cv2.resize(fr, (IMG, IMG), interpolation=cv2.INTER_AREA))
    cap.release(); return out
def _extract_one(v, split):
    d = FR / split / v.parent.name / v.stem.replace(" ", "_"); d.mkdir(parents=True, exist_ok=True)
    if not any(d.glob("*.jpg")):
        for j, fr in enumerate(sample_frames(v, T["har_frames"])):
            cv2.imwrite(str(d / f"{j:03d}.jpg"), fr, [cv2.IMWRITE_JPEG_QUALITY, 95])
    return sorted(d.glob("*.jpg"))
def frames_for(vids, split):
    files, t0 = [None] * len(vids), time.time()
    with ThreadPoolExecutor(max_workers=16) as ex:            # OpenCV releases the GIL while decoding
        futs = {ex.submit(_extract_one, v, split): i for i, v in enumerate(vids)}
        for k, f in enumerate(as_completed(futs), 1):
            files[futs[f]] = f.result()
            if k % 50 == 0 or k == len(vids):
                el = time.time() - t0
                print(f"  [{split}] {k}/{len(vids)} videos, {el/60:.1f} min, ~{el/k*(len(vids)-k)/60:.1f} min left", flush=True)
    pairs, owner = [], []
    for vi, (v, fs) in enumerate(zip(vids, files)):
        for f in fs: pairs.append((str(f), cidx[v.parent.name])); owner.append(vi)
    return pairs, np.array(owner)
with Timer("HAR frame extraction"):
    H_tr, _ = frames_for(v_tr, "train"); H_va, _ = frames_for(v_va, "valid"); H_te, own_te = frames_for(v_te, "test")
print(f"frames: train {len(H_tr)}, valid {len(H_va)}, test {len(H_te)}")

In [ ]:
with Timer("Part D — HAR model"):
    har_model, har_hist, har_info = train_cnn(H_tr, H_va, len(CLASSES), "har")
pr = predict_probs(har_model, H_te, len(CLASSES), "HAR / test")
yt = np.array([y for _, y in H_te]); yh = pr.argmax(1)
vid_prob = np.stack([pr[own_te == i].mean(0) for i in range(len(v_te))])
vid_pred = vid_prob.argmax(1); vid_true = np.array([cidx[v.parent.name] for v in v_te])
RESULTS["D_har"] = dict(classes=CLASSES, videos=dict(train=len(v_tr), valid=len(v_va), test=len(v_te)),
                        frames_per_video=T["har_frames"], video_info=VIDEO_INFO, train=har_info,
                        frame_acc=float((yh == yt).mean()), video_acc=float((vid_pred == vid_true).mean()),
                        video_confusion=confusion_matrix(vid_true, vid_pred, labels=range(len(CLASSES))).tolist())
print(json.dumps({k: v for k, v in RESULTS["D_har"].items() if k != "video_confusion"}, indent=1)); save_results()

fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
ax[0].plot(har_hist["accuracy"], label="train"); ax[0].plot(har_hist["val_accuracy"], label="validation")
ax[0].set_title("HAR accuracy"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(har_hist["loss"], label="train"); ax[1].plot(har_hist["val_loss"], label="validation")
ax[1].set_title("HAR loss"); ax[1].set_xlabel("epoch"); ax[1].legend()
cm = np.array(RESULTS["D_har"]["video_confusion"]); ax[2].imshow(cm, cmap="Blues")
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)): ax[2].text(j, i, cm[i, j], ha="center", va="center", fontsize=8)
ax[2].set_xticks(range(len(CLASSES)), CLASSES, rotation=60, ha="right", fontsize=8)
ax[2].set_yticks(range(len(CLASSES)), CLASSES, fontsize=8); ax[2].set_title("HAR test confusion (videos)")
plt.tight_layout(); plt.savefig(f"{OUT}/figures/D_har.png", dpi=200); plt.show()

## 8. Part E — full dual-stream pipeline on real video
Both trained models process the **same decoded frames** of held-out HAR test videos (batch size 1). Three configurations are timed:

| Configuration | Decoding | Face detector | Detection frequency | HAR stream |
|---|---|---|---|---|
| **naïve** | in the processing loop | Haar cascade on a half-resolution frame | every frame | every frame |
| **optimised (every frame)** | separate reader thread (overlaps with inference) | YuNet on a 640-px-wide frame | every frame | every frame |
| **optimised** | separate reader thread | YuNet on a 640-px-wide frame | every 5th frame, box held in between | every 2nd frame |

For each frame we record the time of every stage (decode, detection, preprocessing, deepfake CNN, HAR CNN). **Throughput** is frames processed per second of wall-clock time; the pipeline is called **real-time** when its throughput is at least the video's native frame rate. Timing loops stop when the mean per-frame time is known to ±`rel_ci` (95% CI), with a minimum and maximum number of frames.

In [ ]:
PIPELINE_PY = r"""
import time, threading, queue, numpy as np, cv2

class Reader(threading.Thread):
    # Decodes frames from a list of videos (looping) into a bounded queue; records decode time per frame.
    def __init__(self, videos, maxsize=16):
        super().__init__(daemon=True); self.videos, self.q, self.stop = videos, queue.Queue(maxsize), False
    def run(self):
        while not self.stop:
            for v in self.videos:
                cap = cv2.VideoCapture(v)
                while not self.stop:
                    t = time.perf_counter(); ok, fr = cap.read(); dt = time.perf_counter() - t
                    if not ok: break
                    self.q.put((fr, dt))
                cap.release()
                if self.stop: break

def frames_inline(videos):
    while True:
        for v in videos:
            cap = cv2.VideoCapture(v)
            while True:
                t = time.perf_counter(); ok, fr = cap.read(); dt = time.perf_counter() - t
                if not ok: break
                yield fr, dt
            cap.release()

class FaceDetector:
    def __init__(self, kind, yunet_path, width):
        self.kind, self.width = kind, width
        if kind == "yunet":
            self.det = cv2.FaceDetectorYN.create(yunet_path, "", (320, 320), 0.6, 0.3, 50); self.size = None
        else:
            self.det = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
    def __call__(self, fr):
        h0, w0 = fr.shape[:2]; s = min(1.0, self.width / w0)
        small = cv2.resize(fr, (int(w0 * s), int(h0 * s))) if s < 1 else fr
        if self.kind == "yunet":
            if self.size != small.shape[:2]:
                self.det.setInputSize((small.shape[1], small.shape[0])); self.size = small.shape[:2]
            _, f = self.det.detect(small)
            if f is None or len(f) == 0: return None
            b = max(f, key=lambda r: r[2] * r[3])[:4]
        else:
            f = self.det.detectMultiScale(cv2.cvtColor(small, cv2.COLOR_BGR2GRAY), 1.2, 5)
            if len(f) == 0: return None
            b = max(f, key=lambda r: r[2] * r[3])
        return [int(v / s) for v in b]

def to_input(img, size=256):
    return (cv2.resize(img, (size, size))[None, ..., ::-1].astype(np.float32) / 255.0)

def run(cfg, videos, df_fn, har_fn, yunet_path, warm, min_frames, max_frames, rel_ci, log=print):
    # Time the dual-stream pipeline. cfg: threaded, detector, det_width, det_every, har_every.
    det = FaceDetector(cfg["detector"], yunet_path, cfg["det_width"])
    reader = None
    if cfg["threaded"]:
        reader = Reader(videos); reader.start(); src = iter(lambda: reader.q.get(), None)
    else:
        src = frames_inline(videos)
    stages = {k: [] for k in ("decode", "detect", "preprocess", "deepfake_cnn", "har_cnn", "total_compute")}
    box, faces, n, t_wall0, totals = None, 0, 0, None, []
    for i, (fr, t_dec) in enumerate(src):
        t0 = time.perf_counter(); rec = {"decode": t_dec}
        if i % cfg["det_every"] == 0:
            t = time.perf_counter(); box = det(fr); rec["detect"] = time.perf_counter() - t
        t = time.perf_counter()
        xd = None
        if box is not None:
            x, y, w, h = box; x, y = max(x, 0), max(y, 0); crop = fr[y:y + h, x:x + w]
            if crop.size: xd = to_input(crop)
        xh = to_input(fr) if i % cfg["har_every"] == 0 else None
        rec["preprocess"] = time.perf_counter() - t
        if xd is not None:
            t = time.perf_counter(); df_fn(xd); rec["deepfake_cnn"] = time.perf_counter() - t
        if xh is not None:
            t = time.perf_counter(); har_fn(xh); rec["har_cnn"] = time.perf_counter() - t
        rec["total_compute"] = time.perf_counter() - t0
        if i == warm: t_wall0 = time.perf_counter()
        if i >= warm:
            n += 1; faces += xd is not None
            for k, v in rec.items(): stages[k].append(v)
            # per-frame cost: compute plus decode when decoding is inline
            totals.append(rec["total_compute"] + (0 if cfg["threaded"] else t_dec))
            if n >= min_frames and n % 25 == 0:
                m = np.mean(totals); half = 1.96 * np.std(totals, ddof=1) / np.sqrt(n)
                done = half / m < rel_ci or n >= max_frames
                log(f"    [stop rule] {n} frames: mean {1000*m:.1f} ms ±{100*half/m:.1f}% "
                    f"(needs ±{100*rel_ci:.0f}%)" + (" -> STOP" if done else ""))
                if done: break
    wall = time.perf_counter() - t_wall0
    if reader: reader.stop = True
    ms = {k: (1000 * float(np.mean(v)) if v else 0.0) for k, v in stages.items()}
    calls = {k: len(v) for k, v in stages.items()}
    # amortised cost per frame of each stage (a stage run on every k-th frame counts 1/k)
    amort = {k: (1000 * float(np.sum(v)) / n if v else 0.0) for k, v in stages.items()}
    return dict(config=cfg, frames=n, frames_with_face=faces, wall_s=wall, throughput_fps=n / wall,
                per_frame_ms_mean=1000 * float(np.mean(totals)), per_frame_ms_p95=1000 * float(np.percentile(totals, 95)),
                stage_ms_per_call=ms, stage_calls=calls, stage_ms_amortised=amort)
"""
open(f"{WORK}/pipeline.py", "w").write(PIPELINE_PY)
sys.path.insert(0, WORK)
import importlib, pipeline; importlib.reload(pipeline)

PIPE_CONFIGS = {
    "naive":                 dict(threaded=False, detector="haar", det_width=10_000, det_every=1, har_every=1),
    "optimised_every_frame": dict(threaded=True, detector=FACE_DETECTOR, det_width=640, det_every=1, har_every=1),
    "optimised":             dict(threaded=True, detector=FACE_DETECTOR, det_width=640, det_every=5, har_every=2),
}
# the naive configuration reproduces the earlier measurement: Haar on a half-resolution frame
PIPE_CONFIGS["naive"]["det_width"] = VIDEO_INFO["width"] // 2
test_videos = [str(v) for v in v_te]
DF_MODEL = MODELS["deepfake_robust"]          # both deepfake models have the same architecture and cost
print("pipeline configurations:", json.dumps(PIPE_CONFIGS, indent=1))

In [ ]:
E = {"video": VIDEO_INFO, "gpu": {}, "cpu": {}}
TM = T["timing"]
if gpus:
    with tf.device("/GPU:0"):
        fd = tf.function(lambda x: DF_MODEL(x, training=False)); fh = tf.function(lambda x: har_model(x, training=False))
        df_fn = lambda x: fd(tf.constant(x)).numpy(); har_fn = lambda x: fh(tf.constant(x)).numpy()
        for name, cfg in PIPE_CONFIGS.items():
            with Timer(f"GPU timing — {name}"):
                r = pipeline.run(cfg, test_videos, df_fn, har_fn, str(YUNET), TM["warm"], TM["min_frames"],
                                 TM["max_frames"], TM["rel_ci"])
            E["gpu"][name] = r
            print(f"  {name}: {r['throughput_fps']:.1f} FPS, per-frame {r['per_frame_ms_mean']:.1f} ms; "
                  f"amortised stages (ms): " + ", ".join(f"{k} {v:.1f}" for k, v in r["stage_ms_amortised"].items()))
RESULTS["E_pipeline"] = E; save_results()

In [ ]:
# CPU timing in fresh processes, so that the thread count is fixed before TensorFlow starts.
DF_MODEL.save_weights(f"{WORK}/df_for_timing.weights.h5"); har_model.save_weights(f"{WORK}/har_for_timing.weights.h5")
json.dump(dict(videos=test_videos, configs=PIPE_CONFIGS, yunet=str(YUNET), timing=TM, n_har=len(CLASSES)),
          open(f"{WORK}/cpu_timing.json", "w"))
CPU_SCRIPT = r"""
import os, sys, json
TH = int(sys.argv[1]); W = sys.argv[2]
os.environ["CUDA_VISIBLE_DEVICES"] = ""; os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import tensorflow as tf, cv2
tf.config.threading.set_intra_op_parallelism_threads(TH); tf.config.threading.set_inter_op_parallelism_threads(TH)
cv2.setNumThreads(TH); sys.path.insert(0, W)
import pipeline
from tensorflow.keras import layers, models
def build(n):
    m = models.Sequential([layers.Input((256, 256, 3))])
    for f in (32, 64, 64, 64, 64, 64):
        m.add(layers.Conv2D(f, 3, activation="relu")); m.add(layers.MaxPooling2D(2))
    m.add(layers.Flatten()); m.add(layers.Dropout(0.25)); m.add(layers.Dense(64, activation="relu"))
    m.add(layers.Dense(n, activation="softmax")); return m
J = json.load(open(f"{W}/cpu_timing.json"))
D = build(2); D.load_weights(f"{W}/df_for_timing.weights.h5")
H = build(J["n_har"]); H.load_weights(f"{W}/har_for_timing.weights.h5")
@tf.function
def fd(x): return D(x, training=False)
@tf.function
def fh(x): return H(x, training=False)
def df_fn(x): return fd(tf.constant(x)).numpy()
def har_fn(x): return fh(tf.constant(x)).numpy()
out = {}
for name, cfg in J["configs"].items():
    print(f"  [CPU {TH} thread(s)] {name} ...", flush=True)
    tm = J["timing"]
    out[name] = pipeline.run(cfg, J["videos"], df_fn, har_fn, J["yunet"], tm["warm"], tm["min_frames"],
                             tm["max_frames"], tm["rel_ci"], log=lambda s: print(s, flush=True))
    print(f"  [CPU {TH} thread(s)] {name}: {out[name]['throughput_fps']:.1f} FPS", flush=True)
json.dump(out, open(f"{W}/cpu_timing_{TH}.json", "w"))
"""
open(f"{WORK}/cpu_timing.py", "w").write(CPU_SCRIPT)
for th in sorted({1, os.cpu_count()}):
    with Timer(f"CPU timing, {th} thread(s)"):
        r = subprocess.run([sys.executable, f"{WORK}/cpu_timing.py", str(th), WORK], text=True)
        assert r.returncode == 0, "CPU timing failed"
    E["cpu"][f"{th}_threads"] = json.load(open(f"{WORK}/cpu_timing_{th}.json"))
RESULTS["E_pipeline"] = E; save_results()

# ---- summary table and stage-breakdown figure ----
fps_video = VIDEO_INFO["fps"] or 30.0
print(f"\nnative frame rate of the test videos: {fps_video:.1f} FPS ({VIDEO_INFO['width']}x{VIDEO_INFO['height']})")
rows = [(f"GPU", n, r) for n, r in E["gpu"].items()] + \
       [(f"CPU {k.split('_')[0]} thr.", n, r) for k, d in E["cpu"].items() for n, r in d.items()]
for dev, name, r in rows:
    verdict = "real-time" if r["throughput_fps"] >= fps_video else "below real-time"
    print(f"{dev:12s} {name:24s} {r['throughput_fps']:6.1f} FPS  ({verdict})")
fig, ax = plt.subplots(figsize=(11, 0.6 * len(rows) + 1.5))
stage_keys = ["decode", "detect", "preprocess", "deepfake_cnn", "har_cnn"]
STAGE_COLOURS = dict(zip(stage_keys, ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3"]))   # same colour per stage in every bar
for i, (dev, name, r) in enumerate(rows):
    left = 0
    for k in stage_keys:
        v = r["stage_ms_amortised"][k]
        ax.barh(i, v, left=left, color=STAGE_COLOURS[k], label=k if i == 0 else None); left += v
ax.set_yticks(range(len(rows)), [f"{d} | {n}" for d, n, _ in rows]); ax.set_xlabel("amortised time per frame (ms)")
ax.axvline(1000 / fps_video, color="k", ls="--", lw=1, label=f"real-time budget ({1000/fps_video:.0f} ms)")
ax.legend(loc="lower right"); ax.set_title("Dual-stream pipeline: time per frame by stage")
plt.tight_layout(); plt.savefig(f"{OUT}/figures/E_pipeline_stages.png", dpi=200); plt.show()

## 9. Part F — export
Writes `results_tables.tex` (LaTeX table rows). `results.json` (updated after every part) and the figures are already in `MyDrive/Outputs/DualStream/results`.

In [ ]:
pct = lambda x: f"{100 * x:.2f}"
L = ["% ===== Part A: deepfake stream, 140k test split (fake = positive) ====="]
for tag, a in RESULTS["A_deepfake"].items():
    m, ci = a["test"], a["ci95"]
    L.append(f"% {tag} ({a['recipe']} recipe): TP={m['TP']} FN={m['FN']} FP={m['FP']} TN={m['TN']}; "
             f"epochs {a['train']['epochs_run']}, best epoch {a['train']['best_epoch']}")
    for k, lab in (("accuracy", "Accuracy"), ("precision", "Precision"), ("recall_sensitivity", "Sensitivity"),
                   ("specificity", "Specificity"), ("f1", "F1-score"), ("auc", "AUC")):
        L.append(f"{tag} & {lab} & {pct(m[k])} & [{pct(ci[k][0])}, {pct(ci[k][1])}] \\\\ \\hline")
L.append("\n% ===== Part B: cross-dataset (no retraining) =====")
for dn, d in RESULTS.get("B_cross", {}).items():
    for tag, vv in d["models"].items():
        for var, row in vv.items():
            a, b = row["threshold_0.5"], row["threshold_val"]
            L.append(f"{dn} & {tag} & {var} & {a['auc']:.3f} & {pct(a['accuracy'])} & {pct(a['recall_sensitivity'])} & "
                     f"{pct(a['specificity'])} & {pct(b['accuracy'])} \\\\ \\hline")
c = RESULTS["C_fusion"]
L.append(f"\n% ===== Part C: fusion, {c['runs']} runs, best single = {c['best_single']}, SVM {c['svm_backend']} =====")
for name, v in c["configs"].items():
    p = v.get("wilcoxon_p"); ps = "--" if p is None else (f"{p:.3f}" if p >= 0.001 else "$<0.001$")
    L.append(f"{name} & {pct(v['mean'])} $\\pm$ {pct(v['sd'])} & {ps} \\\\ \\hline")
d = RESULTS["D_har"]
L.append(f"\n% ===== Part D: HAR, video split {d['videos']}, {d['frames_per_video']} frames/video =====")
L.append(f"Frame-level accuracy & {pct(d['frame_acc'])} \\\\ \\hline\nVideo-level accuracy & {pct(d['video_acc'])} \\\\ \\hline")
E = RESULTS["E_pipeline"]
L.append(f"\n% ===== Part E: pipeline on {E['video']['width']}x{E['video']['height']} video at {E['video']['fps']:.1f} FPS "
         f"({RESULTS['env']['gpu']} / {RESULTS['env']['cpu']}) =====")
for devk, block in [("GPU", E["gpu"])] + [(f"CPU {k.split('_')[0]} thr.", v) for k, v in E["cpu"].items()]:
    for name, r in block.items():
        s = r["stage_ms_amortised"]
        L.append(f"{devk} & {name.replace('_', ' ')} & {s['decode']:.1f} & {s['detect']:.1f} & {s['preprocess']:.1f} & "
                 f"{s['deepfake_cnn']:.1f} & {s['har_cnn']:.1f} & {r['per_frame_ms_mean']:.1f} & {r['throughput_fps']:.1f} \\\\ \\hline")
open(f"{OUT}/results_tables.tex", "w").write("\n".join(L)); save_results()
print("\n".join(L))

print(f"\nAll results, tables and figures are in {OUT} (Google Drive).")

## 10. Part B2 — cross-dataset threshold calibration with a small labelled target sample (stand-alone)
Zero-shot transfer (Part B) keeps a high AUC but the decision threshold learned on the training domain does not transfer. This part measures what a deployment gains from labelling a small sample of the new source: the threshold (not the model) is fixed on `n_cal` labelled Hard Fake images and accuracy is measured on the remaining, disjoint images. Random splits are repeated until the mean accuracy has converged. The cell is self-contained and can be run on its own after the models exist on Drive.

In [ ]:
# =====================================================================================
# Part B2 — Cross-dataset threshold calibration with a small labelled target sample
# -------------------------------------------------------------------------------------
# Stand-alone cell: it mounts Drive, extracts the Hard Fake archive if needed, reloads the two trained
# deepfake models from the Drive checkpoints and writes its results to Drive. No training.
#
# Protocol (fixed in advance):
#   * Hard Fake vs Real Faces (1,289 images) is split at random, stratified by class, into a CALIBRATION
#     sample of n_cal labelled images and a disjoint EVALUATION set (all remaining images).
#   * The decision threshold is chosen on the calibration sample only (maximum balanced accuracy);
#     the model itself is not changed. Metrics are computed on the evaluation set only.
#   * n_cal in {20, 50, 100, 200, half}. For each n_cal the random split is repeated; the loop stops when
#     the 95% CI half-width of the mean evaluation accuracy is below 0.25 percentage points
#     (minimum 30 repetitions, maximum 1000), and prints its stopping status.
#   * Reported for reference: zero-shot (threshold 0.5) on the same images.
# =====================================================================================
import os, sys, json, time, shutil, zipfile
from pathlib import Path
import numpy as np

DRIVE_ROOT = "/content/drive/MyDrive"
HARD_ZIP   = f"{DRIVE_ROOT}/Datasets/HAR/Hard_Fake_Real_Faces.zip"
CKPT       = f"{DRIVE_ROOT}/Outputs/DualStream/checkpoints"
OUT        = f"{DRIVE_ROOT}/Outputs/DualStream/results"
WORK       = "/content/work"                      # temporary extracted images only
SEED, IMG  = 42, 256
N_CAL      = [20, 50, 100, 200, "half"]
MIN_REP, MAX_REP, CI_TOL = 30, 1000, 0.0025

# ---- mount Drive first ----
try:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        if os.path.exists("/content/drive"): shutil.rmtree("/content/drive")
        drive.mount("/content/drive")
except ImportError:
    pass
assert os.path.exists(HARD_ZIP), f"missing {HARD_ZIP}"
Path(OUT).mkdir(parents=True, exist_ok=True)

# ---- data (extract once to /content/work) ----
dst = Path(WORK) / "ext_HardFakevsRealFaces"
if not (dst / ".extracted").exists():
    print("extracting Hard Fake vs Real Faces ...", flush=True)
    dst.mkdir(parents=True, exist_ok=True); zipfile.ZipFile(HARD_ZIP).extractall(dst); (dst / ".extracted").touch()
def label_from_path(p):
    for s in reversed([s.lower() for s in Path(p).parts[:-1]]):
        if "real" in s and "fake" not in s: return 0
        if "fake" in s and "real" not in s: return 1
    return None
pairs = [(str(p), label_from_path(p)) for p in sorted(dst.rglob("*"))
         if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}]
pairs = [(p, y) for p, y in pairs if y is not None]
y = np.array([t for _, t in pairs])
print(f"{len(pairs)} images, {int(y.sum())} fake, {int((1 - y).sum())} real")

# ---- models (same architecture as Parts A-D; float32 weights from Drive) ----
import tensorflow as tf
from tensorflow.keras import layers, models
tf.keras.mixed_precision.set_global_policy("float32")
def build_cnn(n_out):
    m = models.Sequential([layers.Input((IMG, IMG, 3))])
    for f in (32, 64, 64, 64, 64, 64):
        m.add(layers.Conv2D(f, 3, activation="relu")); m.add(layers.MaxPooling2D(2))
    m.add(layers.Flatten()); m.add(layers.Dropout(0.25)); m.add(layers.Dense(64, activation="relu"))
    m.add(layers.Dense(n_out, activation="softmax", dtype="float32")); return m
def load_img(path):
    x = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    return tf.cast(tf.cast(tf.image.resize(x, (IMG, IMG)), tf.uint8), tf.float32) / 255.0   # identical to training
ds = tf.data.Dataset.from_tensor_slices([p for p, _ in pairs]).map(load_img, tf.data.AUTOTUNE).batch(128)

SCORES = {}
for tag in ("deepfake", "deepfake_robust"):
    m = build_cnn(2); m.load_weights(f"{CKPT}/{tag}_final.weights.h5")
    SCORES[tag] = np.concatenate([m(x, training=False).numpy()[:, 1] for x in ds])
    print(f"{tag}: scores computed", flush=True)

# ---- metrics ----
from sklearn.metrics import roc_auc_score
def metrics(yt, s, thr):
    yh = (s >= thr).astype(int)
    tp = int(((yt == 1) & (yh == 1)).sum()); tn = int(((yt == 0) & (yh == 0)).sum())
    fp = int(((yt == 0) & (yh == 1)).sum()); fn = int(((yt == 1) & (yh == 0)).sum())
    sens, spec = tp / max(tp + fn, 1), tn / max(tn + fp, 1); prec = tp / max(tp + fp, 1)
    return dict(accuracy=(tp + tn) / len(yt), sensitivity=sens, specificity=spec,
                balanced_accuracy=(sens + spec) / 2, f1=2 * prec * sens / max(prec + sens, 1e-12))
GRID = np.linspace(0.0005, 0.9995, 1999)
def best_threshold(yc, sc):
    ba = [((sc[yc == 1] >= t).mean() + (sc[yc == 0] < t).mean()) / 2 for t in GRID]
    return float(GRID[int(np.argmax(ba))])

idx_fake, idx_real = np.where(y == 1)[0], np.where(y == 0)[0]
RES = {"protocol": dict(n_images=len(y), n_fake=int(y.sum()), n_cal=[str(n) for n in N_CAL],
                        threshold_rule="max balanced accuracy on calibration sample",
                        stop_rule=f"95% CI half-width of mean accuracy < {100*CI_TOL:.2f} pts, min {MIN_REP}, max {MAX_REP}"),
       "models": {}}
for tag, s in SCORES.items():
    RES["models"][tag] = {"zero_shot_0.5": metrics(y, s, 0.5), "auc": float(roc_auc_score(y, s)), "calibrated": {}}
    print(f"\n=== {tag}: AUC {RES['models'][tag]['auc']:.4f}, zero-shot accuracy "
          f"{100*RES['models'][tag]['zero_shot_0.5']['accuracy']:.2f}% ===", flush=True)
    for n_cal in N_CAL:
        if n_cal != "half" and n_cal > len(y) // 2:
            print(f"  n_cal={n_cal}: skipped (more than half of the images)"); continue
        rng = np.random.default_rng(SEED); reps = []; thrs = []; t0 = time.time()
        for r in range(1, MAX_REP + 1):
            k = (len(y) // 2) if n_cal == "half" else n_cal
            kf = int(round(k * len(idx_fake) / len(y)))                       # stratified by class
            cf = rng.choice(idx_fake, kf, replace=False); cr = rng.choice(idx_real, k - kf, replace=False)
            cal = np.r_[cf, cr]; ev = np.setdiff1d(np.arange(len(y)), cal)
            thr = best_threshold(y[cal], s[cal]); thrs.append(thr)
            reps.append(metrics(y[ev], s[ev], thr))
            if r >= MIN_REP and r % 10 == 0:
                a = np.array([m["accuracy"] for m in reps]); half = 1.96 * a.std(ddof=1) / np.sqrt(r)
                if half < CI_TOL or r == MAX_REP:
                    print(f"  n_cal={str(n_cal):>4}: [stop rule] {r} splits, CI half-width {100*half:.2f} pts -> STOP "
                          f"({time.time()-t0:.0f} s)", flush=True)
                    break
        keys = reps[0].keys()
        summ = {k: dict(mean=float(np.mean([m[k] for m in reps])), sd=float(np.std([m[k] for m in reps], ddof=1)),
                        p5=float(np.percentile([m[k] for m in reps], 5)), p95=float(np.percentile([m[k] for m in reps], 95)))
                for k in keys}
        summ["threshold"] = dict(mean=float(np.mean(thrs)), sd=float(np.std(thrs, ddof=1)))
        summ["splits"] = len(reps)
        RES["models"][tag]["calibrated"][str(n_cal)] = summ
        print(f"  n_cal={str(n_cal):>4}: accuracy {100*summ['accuracy']['mean']:.2f} ± {100*summ['accuracy']['sd']:.2f}%  "
              f"(5–95%: {100*summ['accuracy']['p5']:.1f}–{100*summ['accuracy']['p95']:.1f}), "
              f"sens {100*summ['sensitivity']['mean']:.1f}%, spec {100*summ['specificity']['mean']:.1f}%, "
              f"threshold {summ['threshold']['mean']:.3f}", flush=True)

json.dump(RES, open(f"{OUT}/B2_calibration.json", "w"), indent=2)
# also merge into results.json, if present
rj = Path(OUT) / "results.json"
if rj.exists():
    R = json.load(open(rj)); R["B2_calibration"] = RES; json.dump(R, open(rj, "w"), indent=2)
print(f"\nsaved: {OUT}/B2_calibration.json (and merged into results.json)")